REPD data where it contains windfarm metadata

In [ ]:
import pandas as pd

#obtaining windfarm metadata
df = pd.read_csv(
    "../data/raw/repd/REPD_2026.csv",
    encoding="latin1"
)

In [ ]:
df.columns.tolist()

In [ ]:
df.head()

We dont need all the variables so filtering them out by:
- Technology Type
    - Wind Onshore and Wind Offshore
        - Development Status == Operational

Here we are filtering out just the windfarms which are opertaional

In [ ]:
df["Technology Type"].value_counts()

In [ ]:
#
wind_df = df[
    df["Technology Type"].isin(
        [
            "Wind Onshore",
            "Wind Offshore"
        ]
    )
]

In [ ]:
wind_df = wind_df[
    wind_df["Development Status"] == "Operational"
]

In [ ]:
wind_df["Development Status"].value_counts()

In [ ]:
#clean windfarm metadata only
repd_clean = wind_df[
    [
        "Site Name",
        "Technology Type",
        "Installed Capacity (MWelec)",
        "Turbine Capacity (MW)",
        "No. of Turbines",
        "Height of Turbines (m)",
        "Development Status",
        "Region",
        "Country",
        "X-coordinate",
        "Y-coordinate"
    ]
]

In [ ]:
repd_clean.info()

In [ ]:
repd_clean.isna().sum()

In [ ]:
repd_clean["Site Name"].duplicated().sum()

In [ ]:
duplicates = repd_clean[
    repd_clean["Site Name"].duplicated(keep=False)
]

duplicates.sort_values("Site Name")

In [ ]:
repd_clean[
    repd_clean["Site Name"]
    == "Hagshaw Hill Wind Farm"
]

In [ ]:
repd_clean[
    repd_clean.duplicated(
        subset=[
            "Site Name",
            "Technology Type",
            "Installed Capacity (MWelec)",
            "Turbine Capacity (MW)",
            "No. of Turbines",
            "Development Status"
        ],
        keep=False
    )
]

In [ ]:
repd_clean = repd_clean.drop_duplicates(
    subset=[
        "Site Name",
        "Technology Type",
        "Installed Capacity (MWelec)",
        "Turbine Capacity (MW)",
        "No. of Turbines",
        "Development Status"
    ]
)

In [ ]:
repd_clean.shape

In [ ]:
repd_clean[
    repd_clean["Site Name"]
    == "Hagshaw Hill Wind Farm"
]

In [ ]:
import pandas as pd

repd_clean = pd.read_parquet(
    "../data/interim/repd_processed/wind_farms.parquet"
)

REPD data is renewable energy metadata, after filtering only windfarms saved as wind_farm

In [ ]:
repd_clean.info()

In [ ]:
import geopandas as gpd

#changing ers to eu standadrd to join with era5 dataset
wind_gdf = gpd.GeoDataFrame(
    repd_clean,
    geometry=gpd.points_from_xy(
        repd_clean["X-coordinate"],
        repd_clean["Y-coordinate"]
    ),
    crs="EPSG:27700"
)

In [ ]:
wind_gdf = wind_gdf.to_crs("EPSG:4326")

In [ ]:
wind_gdf.head()

extra

In [ ]:
import rasterio

with rasterio.open(
    "../data/interim/terrain_processed/uk_dem.tif"
) as dem:

    coords = [
        (point.x, point.y)
        for point in wind_gdf.geometry
    ]

    elevations = [
        val[0]
        for val in dem.sample(coords)
    ]

wind_gdf["Elevation_m"] = elevations

In [ ]:
wind_gdf[
    [
        "Site Name",
        "Elevation_m"
    ]
]

In [ ]:
wind_gdf["Elevation_m"].describe()

In [ ]:
wind_gdf.info()

In [ ]:
wind_gdf.head()

In [ ]:
wind_gdf.to_csv(
    "../data/interim/terrain_processed/wind_farm_elevation.csv",
    index=False
)